# 09 Buuin ang Bantay Santa Luz dashboard — notebook

30 Setyembre 2026 na. Gusto ng City Mayor ng **isang screen** para sa Local Health Board:
ilang kaso, saan, pataas ba o pababa, at ano ang gagawin natin. Sa notebook na ito, bubuo si
Maya ng simpleng bersiyon ng screen na iyon sa tulong ni Tala, hakbang-hakbang:

1. kuwentahin ang mga headline number (KPI);
2. gumawa ng tatlong Plotly chart: epidemic curve, mapa, at dengue laban sa threshold nito;
3. ilagay ang lahat sa **iisang self-contained na HTML file** gamit ang isang maliit na template;
4. patakbuhin ang **publish check** ni Doc Mon bago lumabas ng opisina ang anuman.

Kinuwenta ang lahat dito mula sa synthetic na datos ng Santa Luz. Kathang-isip ang Santa Luz,
ang mga tao nito at ang mga numero nito. Mapupunta sa temporary folder ng computer mo ang
file na isusulat natin, hindi sa aklat.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    sys.path.insert(0, '/content/python4epi/src')
    !pip install -q -e .

## Hakbang 1 · I-load ang datos

Apat na file ang kailangan natin: ang malinis at de-identified na line list, ang populasyon
ng mga barangay, ang lingguhang bilang ng dengue at ang mapa ng mga barangay (GeoJSON). Walang
pangalan o address sa alinman sa mga ito — tinanggal na iyon sa kabanata 03.

In [ ]:
import importlib.util
import json
import re
import tempfile
from pathlib import Path
from string import Template

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

DATA = Path("data/synthetic/ph")
ll = pd.read_csv(DATA / "santa_luz_linelist.csv", parse_dates=["date_onset"])
barangays = pd.read_csv(DATA / "santa_luz_barangays.csv")
weekly = pd.read_csv(DATA / "santa_luz_dengue_weekly.csv")
geojson = json.loads((DATA / "santa_luz_barangays.geojson").read_text(encoding="utf-8"))
AS_OF = "30 Setyembre 2026"
AGE_GROUPS = ["0-4", "5-14", "15-24", "25-44", "45-64", "65+"]

print(len(ll), "kaso ·", len(barangays), "barangay ·", len(weekly), "linggo ng bilang ng dengue")

## Hakbang 2 · Ilan? Ang mga headline number

Tinanong ni Maya si Tala:

> *May pandas DataFrame akong `ll`, isang row bawat kaso, na may mga column na `case_id`,
> `disease` (`leptospirosis` o `dengue`), `outcome` (`alive` o `died`) at `hospitalised`
> (0/1). Mga gawa-gawang row: `X-1, dengue, alive, 0` · `X-2, leptospirosis, died, 1` ·
> `X-3, dengue, alive, 1`. Sumulat ng function na `compute_kpis(ll)` na nagbabalik ng
> bilang ng kaso, namatay, na-ospital at ng case fatality rate sa % (isang decimal) para sa
> bawat sakit at para sa buong lungsod.*

Ito ang ibinalik niya:

In [ ]:
def compute_kpis(ll: pd.DataFrame) -> pd.DataFrame:
    """Kaso, namatay, na-ospital at CFR (%) bawat sakit, at isang row ng kabuuan ng lungsod."""
    kpis = ll.groupby("disease").agg(
        cases=("case_id", "size"),
        deaths=("outcome", lambda s: (s == "died").sum()),
        hospitalised=("hospitalised", "sum"),
    )
    kpis.loc["total"] = kpis.sum()
    kpis["cfr_pct"] = (100 * kpis["deaths"] / kpis["cases"]).round(1)
    return kpis


kpis = compute_kpis(ll)
kpis

**I-check natin.** Dalawang mabilisang check: patakbuhin ang function sa tatlong row na kaya nating
bilangin nang mano-mano, at bilangin ang namatay sa ikalawa at ibang paraan. Dapat *kaso* ang
denominator ng CFR, at dapat tumugma ang row ng kabuuan.

In [ ]:
tiny = pd.DataFrame({"case_id": ["X-1", "X-2", "X-3"],
                     "disease": ["dengue", "leptospirosis", "dengue"],
                     "outcome": ["alive", "died", "alive"],
                     "hospitalised": [0, 1, 1]})
tiny_kpis = compute_kpis(tiny)
assert tiny_kpis.loc["dengue", "cases"] == 2 and tiny_kpis.loc["leptospirosis", "cfr_pct"] == 100
assert kpis.loc["total", "deaths"] == (ll["outcome"] == "died").sum()
assert kpis.loc["total", "cases"] == len(ll)
print("Pasado ang mga check.")

## Hakbang 3 · Ang patakaran sa privacy na gagamitin natin kahit saan

Napupunta ang dashboard sa mga taong nasa labas ng CESU. Ang bilang na **1 hanggang 4** sa
isang maliit na grupo (isang barangay, isang linggo, isang age group) ay puwedeng magturo sa
isang kapitbahay. Kaya bago iguhit o i-print ang anuman, nagiging **"<5"** ang bilang na 1
hanggang 4 at hindi inilalabas ang rate nila. Nananatiling zero ang zero: wala itong
inilalarawang tao. Buong inilalabas ang kabuuan ng lungsod para sa buong panahon (ang mga
KPI), gaya ng sa mga karaniwang bulletin.

In [ ]:
def suppress(counts: pd.Series | pd.DataFrame, threshold: int = 5) -> pd.Series | pd.DataFrame:
    """Palitan ng '<5' ang bilang mula 1 hanggang threshold - 1 (nananatiling 0 ang zero)."""
    small = (counts > 0) & (counts < threshold)
    return counts.astype(object).mask(small, f"<{threshold}")


suppress(pd.Series([0, 1, 4, 5, 12]))

## Hakbang 4 · Kailan? Ang epidemic curve

Mga kaso ayon sa **linggo** ng onset (ISO week), magkapatong ang dalawang sakit. Sapat ang
laki ng linggo para hindi mabunyag ang eksaktong petsa ng onset ng sinuman. Ang linggong may
1–4 na kaso ng isang sakit ay iniiwang blangko at may label na "<5" sa hover text. Minamarkahan
natin ang Bagyong Amihan (18 Hulyo, sa week 29).

In [ ]:
ll["week"] = ll["date_onset"].dt.isocalendar().week
curve = pd.crosstab(ll["week"], ll["disease"]).reindex(range(28, 40), fill_value=0)
public_curve = curve.mask((curve > 0) & (curve < 5))      # 1-4: hindi iginuguhit
labels = [f"W{w}" for w in curve.index]

fig_curve = go.Figure()
for disease, colour in [("leptospirosis", "#0038A8"), ("dengue", "#E8590C")]:
    fig_curve.add_bar(x=labels, y=public_curve[disease], name=disease.title(),
                      marker_color=colour, customdata=suppress(curve[disease]),
                      hovertemplate="%{x}: %{customdata} kaso<extra></extra>")
fig_curve.add_annotation(x="W29", y=curve.sum(axis=1).max(), text="Bagyong Amihan, 18 Hul",
                         showarrow=True, arrowhead=2, ay=-30)
fig_curve.update_layout(barmode="stack", bargap=0.1, height=380,
                        title="Mga kaso ayon sa linggo ng pagsisimula ng sintomas, 2026",
                        xaxis_title="ISO week ng onset", yaxis_title="Kaso")
fig_curve

In [ ]:
# Check: nasa curve ang bawat kasong may petsa (2 kaso ang walang petsa ng onset)
assert curve.to_numpy().sum() == ll["date_onset"].notna().sum() == len(ll) - 2
print(curve.loc[30:34, "leptospirosis"].tolist(), "<- leptospirosis W30-W34; sa README: 35, 108, 68, 15, 2")

## Hakbang 5 · Saan? Mapa ng dengue attack rate

Nakakalinlang ang bilang sa mapa, dahil mas maraming kaso ang malalaking barangay. Ang
iminamapa natin ay ang **attack rate**: kaso ÷ populasyon × 10,000. Ang barangay na may 1–4
na kaso ng dengue ay nagpapakita ng "<5" at **walang rate** (magkasamang ibabalik ng rate at
ng populasyon ang bilang).

In [ ]:
counts = (ll.groupby(["barangay_id", "disease"]).size().unstack(fill_value=0)
          .reindex(barangays["barangay_id"], fill_value=0))
table = barangays.set_index("barangay_id")[["barangay", "band", "population"]].join(counts)
table["dengue_rate"] = (10_000 * table["dengue"] / table["population"]).round(1)
table.loc[table["dengue"].between(1, 4), "dengue_rate"] = None   # huwag ilabas ang rate ng maliliit na cell
table["dengue_shown"] = suppress(table["dengue"])

fig_map = px.choropleth(
    table.reset_index(), geojson=geojson, locations="barangay_id",
    featureidkey="properties.barangay_id", color="dengue_rate",
    color_continuous_scale="Oranges", hover_name="barangay",
    hover_data={"barangay_id": False, "dengue_shown": True, "dengue_rate": True},
    labels={"dengue_rate": "bawat 10,000", "dengue_shown": "kaso"},
    title="Dengue attack rate bawat 10,000 (abo: wala pang 5 kaso, itinago ang rate)")
fig_map.update_geos(fitbounds="locations", visible=False)
fig_map.update_layout(height=420, margin={"l": 0, "r": 0, "t": 50, "b": 0})
fig_map

In [ ]:
# Ikumpara sa README ang pinakamataas na rate: Talaba 43.4, Buhangin 35.5
print(table.sort_values("dengue_rate", ascending=False)[["barangay", "dengue_shown", "dengue_rate"]].head(3))
print("Mga barangay na itinago ang dengue rate:", table["dengue_rate"].isna().sum())

## Hakbang 6 · Pataas o pababa? Dengue laban sa epidemic threshold

Ang threshold ng bawat ISO week ay ang **mean + 2 standard deviation** ng linggong iyon at ng
dalawang katabi nito sa mga karaniwang taon 2022–2025 (kabanata 06). Iginuguhit natin ang 2026
mula week 22.

In [ ]:
base = weekly[weekly["iso_year"].between(2022, 2025)]


def threshold_for(week: int) -> float:
    """Mean + 2 SD ng mga linggong week-1, week, week+1 sa mga baseline year."""
    window = [(week - 2) % 52 + 1, week, week % 52 + 1]
    values = base.loc[base["iso_week"].isin(window), "cases"]
    return values.mean() + 2 * values.std()


y2026 = weekly[(weekly["iso_year"] == 2026) & (weekly["iso_week"] >= 22)].copy()
y2026["threshold"] = y2026["iso_week"].map(threshold_for)
above = y2026[y2026["cases"] > y2026["threshold"]]
print(len(above), "linggong lampas sa threshold, ang una:", above["morbidity_week"].iloc[0])

shown = y2026["cases"].mask(y2026["cases"].between(1, 4))
fig_thr = go.Figure()
fig_thr.add_bar(x=y2026["morbidity_week"], y=shown, name="Kaso ng dengue", marker_color="#E8590C",
                customdata=suppress(y2026["cases"]),
                hovertemplate="%{x}: %{customdata} kaso<extra></extra>")
fig_thr.add_scatter(x=y2026["morbidity_week"], y=y2026["threshold"], name="Epidemic threshold",
                    mode="lines", line={"color": "#CE1126", "dash": "dash", "shape": "hvh"})
fig_thr.update_layout(height=360, title="Lingguhang kaso ng dengue laban sa epidemic threshold, 2026",
                      yaxis_title="Kaso")
fig_thr

In [ ]:
# Ikumpara sa README: unang lumampas noong 2026-W32, 33 kaso laban sa threshold na 17.5
w32 = y2026.set_index("morbidity_week").loc["2026-W32"]
assert len(above) == 8 and w32["cases"] == 33 and round(w32["threshold"], 1) == 17.5
print("Pasado ang mga check sa threshold. Paalala: kulang pa ang W39 (reporting delay).")

## Hakbang 7 · Iisang self-contained na HTML file

Humingi si Maya kay Tala ng "maliit na HTML template na may title, petsang *as of*, mga KPI
tile, ang tatlong chart at isang seksiyon ng data notes, na nakasulat bilang IISANG file na
gumagana offline". Pinupunan ng `string.Template` ng Python ang mga `$placeholder`. Ginagawang
HTML fragment ng `fig.to_html(full_html=False, ...)` ang bawat chart; ang una ang may dala ng
mismong Plotly library (`include_plotlyjs=True`), kaya hindi kailangan ng internet ang file.

In [ ]:
PAGE = Template("""<!DOCTYPE html>
<html lang="fil"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>$title</title>
<style>
  body { font-family: system-ui, sans-serif; max-width: 1100px; margin: 0 auto; padding: 12px;
         background: #fffbf2; color: #1c1b29; }
  header { background: #0038A8; color: #FCD116; padding: 10px 16px; border-radius: 12px; }
  header p { color: #fff; margin: 0; }
  .kpis { display: flex; flex-wrap: wrap; gap: 10px; margin: 12px 0; }
  .kpi { flex: 1 1 140px; background: #fff; border: 2px solid #eadcc0; border-radius: 12px;
         padding: 8px 12px; }
  .kpi b { display: block; font-size: 28px; color: #0038A8; }
  .notes { font-size: 14px; color: #4d4a5e; }
</style></head><body>
<header><h1>$title</h1><p>Datos hanggang $as_of · synthetic na datos para sa pagsasanay</p></header>
<section class="kpis">$kpis</section>
$figures
<section class="notes"><h2>Mga tala sa datos</h2><ul>$notes</ul></section>
</body></html>""")

tiles = [("Kaso", kpis.loc["total", "cases"]),
         ("Namatay", kpis.loc["total", "deaths"]),
         ("CFR · leptospirosis", f'{kpis.loc["leptospirosis", "cfr_pct"]}%'),
         ("CFR · dengue", f'{kpis.loc["dengue", "cfr_pct"]}%'),
         ("Linggong lampas sa dengue threshold", len(above))]
notes = ["Kathang-isip ang Santa Luz City, at synthetic ang lahat ng datos nito, ginawa para sa pagtuturo.",
         "Aggregate na bilang lang: walang pangalan, address, case ID o eksaktong petsa ng mga kaso.",
         "Ipinapakitang <5 ang bilang na 1-4 sa anumang barangay o linggo, at itinatago ang rate nila.",
         "Mga kaso ayon sa linggo ng onset ng sintomas; kulang pa ang huling linggo (reporting delay)."]

figures = [fig.to_html(full_html=False, include_plotlyjs=(i == 0), config={"responsive": True})
           for i, fig in enumerate([fig_curve, fig_map, fig_thr])]
page_html = PAGE.substitute(
    title="Bantay Santa Luz",
    as_of=AS_OF,
    kpis="".join(f'<div class="kpi">{label}<b>{value}</b></div>' for label, value in tiles),
    figures="\n".join(figures),
    notes="".join(f"<li>{note}</li>" for note in notes),
)
out = Path(tempfile.gettempdir()) / "bantay_santa_luz_simple.html"
out.write_text(page_html, encoding="utf-8")
print(f"Naisulat ang {out} ({out.stat().st_size / 1e6:.1f} MB). Buksan ito sa browser, tapos sa phone.")

## Hakbang 8 · Ang publish check ni Doc Mon

Bago lumabas ng CESU ang file, dumadaan si Doc Mon sa isang checklist. May ilang item na kayang
suriin ng code; gawin silang function para mapatakbo mo ito sa bawat bersiyon:

- aggregate lang — walang case ID, walang column ng line list;
- naka-suppress ang maliliit na cell — walang table cell na 1–4;
- may petsang *as of* at mga tala sa datos na nagsasabi kung ano ang mga numero.

(May mga item na tao pa rin ang susuri: walang pangalan o eksaktong address sa teksto, tapat
na axis, na-test sa phone.)

In [ ]:
def publish_check(page: str) -> list[str]:
    """Ibalik ang mga problemang haharangin ni Doc Mon (walang laman na listahan = OK i-publish)."""
    problems = []
    if re.search(r"SLC-\d{4}", page):
        problems.append("may nakitang case ID: bawal i-publish ang line-level data")
    for column in ["case_id", "date_onset", "date_report"]:
        if column in page:
            problems.append(f"may nakitang column ng line list: {column!r}")
    small = re.findall(r"<td>\s*[1-4]\s*</td>", page)
    if small:
        problems.append(f"{len(small)} table cell na may 1-4 kaso: i-suppress ang mga ito")
    for must in ["Datos hanggang", "Mga tala sa datos", "synthetic"]:
        if must not in page:
            problems.append(f"kulang: {must!r}")
    return problems


print(publish_check(page_html) or "PASADO: handa na para sa huling pagsusuri ni Doc Mon")

### Nagkamali si Tala

Humingi ng "mas detalyado" ang staff ng Mayor, kaya pinakiusapan ni Maya si Tala na *magdagdag
ng table ng kaso ayon sa barangay at age group, at ng button para i-download ang datos*. Mukhang
maayos ang code ni Tala:

```python
detail = pd.crosstab(ll["barangay"], ll["age_group"])          # hilaw na bilang, kasama ang mga 1 at 2
download = f"<script>const DATA = {ll.to_json(orient='records')};</script>"  # ang buong line list!
bad_page = page_html.replace("</body>", detail.to_html() + download + "</body>")
```

Patakbuhin natin ito — **sa memory lang**, hindi kailanman ise-save — at ipasa sa publish check.

In [ ]:
detail = pd.crosstab(ll["barangay"], ll["age_group"])
download = f"<script>const DATA = {ll.to_json(orient='records', date_format='iso')};</script>"
bad_page = page_html.replace("</body>", detail.to_html() + download + "</body>")
for problem in publish_check(bad_page):
    print("HARANG:", problem)

Dalawang seryosong problema. May ilang table cell na **1** o **2**: sa barangay na may ilang
libong tao, ang "isang kasong 65 pataas" ay taong kayang pangalanan ng mga kapitbahay, hindi
statistic. At inilagay ng "download" ang **bawat row ng line list** — case ID, edad, eksaktong
petsa ng onset — sa loob ng pahina, kung saan mababasa ito ng kahit sino gamit ang *View
source*. Ang solusyon: aggregate lang ang i-publish, naka-suppress.

In [ ]:
safe_detail = suppress(pd.crosstab(ll["band"], ll["age_group"])[AGE_GROUPS])  # mas malawak na grupo, tapos i-suppress
fixed_page = page_html.replace("</body>", safe_detail.to_html() + "</body>")
print(publish_check(fixed_page) or "PASADO")
safe_detail

## Hakbang 9 · Ang buong dashboard ng site ay parehong ideya, mas pinalawak

Ang [Bantay Santa Luz dashboard](https://ancientsky.github.io/python4epi/ph/fil/dashboard/) sa
site ng aklat na ito ay binuo ng `scripts/build_ph_dashboard.py` sa parehong mga hakbang:
`load_data()`, `compute_kpis()`, `suppress_small_cells()`, `render_html()`. Iginuguhit nito ang
mga chart gamit ang isang maliit na script na isinulat mismo sa halip na Plotly, kaya wala pang
0.2 MB ang file. May isa pang patakaran ang `suppress_small_cells()` nito: kapag naka-publish
ang kabuuan ng isang column at iisang cell lang ng column na iyon ang nakatago, may itinatagong
pangalawang cell (`x`), para walang makakuha ulit sa maliit na cell sa pamamagitan ng
pagbabawas.

Bahagi ng repository ang script (kinlone ito ng Colab cell), kaya puwede natin itong i-import:

In [ ]:
spec = importlib.util.spec_from_file_location("build_ph_dashboard", "scripts/build_ph_dashboard.py")
dashboard = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dashboard)

full = dashboard.compute_kpis(dashboard.load_data())
assert full["cases"]["total"] == kpis.loc["total", "cases"]
assert full["cfr_pct"]["leptospirosis"] == kpis.loc["leptospirosis", "cfr_pct"]
print("Parehong KPI ng sa atin:", full["cases"], full["cfr_pct"])

dengue = ll[ll["disease"] == "dengue"]
dashboard.suppress_small_cells(pd.crosstab(dengue["age_group"], dengue["sex"]).reindex(AGE_GROUPS))

In [ ]:
# Buuin ang buong dashboard sa temporary folder mo (mga isang segundo lang)
page = dashboard.build_dashboard("fil", Path(tempfile.gettempdir()) / "bantay_full")
print(page, f"{page.stat().st_size / 1e3:.0f} kB")
print(publish_check(page.read_text(encoding="utf-8")) or "PASADO")

## Ikaw naman

1. **Magdagdag ng tile.** Ilagay sa `tiles` ang bilang ng na-ospital na kaso, buuin ulit ang
   `page_html` at patakbuhin ulit ang publish check. *Pahiwatig:* `kpis.loc["total", "hospitalised"]`.
2. **Imapa ang ibang sakit.** Ipakita sa `fig_map` ang attack rate ng leptospirosis.
   *Pahiwatig:* kopyahin ang tatlong linyang `dengue_…` sa Hakbang 5 at palitan ang `dengue` ng
   `leptospirosis`. Aling bahagi ng lungsod ang nagliliwanag?
3. **Mas mahigpit na patakaran.** May mga opisinang nagsu-suppress ng bilang na mas mababa sa
   10. Ilang barangay ang mawawalan ng dengue rate kapag `threshold=10`? *Pahiwatig:*
   `(suppress(table["dengue"], 10) == "<10").sum()`.

In [ ]:
# Ikaw naman 1: magdagdag ng tile ng na-ospital

In [ ]:
# Ikaw naman 2: imapa ang leptospirosis

In [ ]:
# Ikaw naman 3: mas mahigpit na suppression rule

## Bago mo ito ipadala

Buksan ang HTML file **sa phone** (ipadala mo sa sarili mo, hindi sa group chat), basahin nang
malakas nang minsan ang bawat numero, at hingin kay Doc Mon — isang tao — ang huling oo. Ang
code ang nakakahuli ng mga mekanikal na pagkakamali; ang mga tao ang nakakahuli sa iba pa.